
1) CLIP embeddings: (Contrastive Language-Image Pretraining)

![](https://www.dailydoseofds.com/content/images/2024/12/image.png)


2) Multimodal prompting:


![](https://www.dailydoseofds.com/content/images/2024/12/Screenshot-2024-12-03-at-6.38.55-PM.png)


## 1) CLIP embeddings

CLIP (Contrastive Language–Image Pretraining) is a model developed by OpenAI that creates a **shared representation space for text and images**.

![](https://www.dailydoseofds.com/content/images/2024/12/image-1.png)

Unlike traditional models that handle text or images in isolation, **CLIP allows us to compare and reason about text and images togethe**r, which makes it a key component of multimodal systems like Retrieval-Augmented Generation (RAG).


### Motivating task

As an ML engineer, you are responsible for building a face unlock system.

Let’s look through some possible options:

#### Option 1) How about a simple binary classification model?

Output `1` if the true user is opening the mobile; `0` otherwise.

![](https://www.dailydoseofds.com/content/images/2024/12/image-2.png)

Initially, you can ask the user to input facial data to train the model.
But that’s where you identify the problem.All samples will belong to “Class 1.”

![](https://www.dailydoseofds.com/content/images/2024/12/image-3.png)

Now, you can’t ask the user to find someone to volunteer for “Class 0” samples since it’s too much hassle for them.

Not only that, you also need diverse “Class 0” samples. Samples from just one or two faces might not be sufficient.

The next possible solution you think of is…

**Maybe ship some negative samples (Class 0) to the device to train the model.**

![](https://www.dailydoseofds.com/content/images/2024/12/image-4.png)

Might work.But then you realize another problem:What if another person wants to use the same device?
**Since all new samples will belong to the “new face” during adaptation, what if the model forgets the first face?**

![](https://www.dailydoseofds.com/content/images/2024/12/image-5.png)

----------

#### Option 2) How about transfer learning?


This is extremely useful when:
-   **The task of interest has less data.**
-   **But a related task has abundant data.**

This is how you think it could work in this case:

-   Train a neural network model (base model) on some related task → This will happen before shipping the model to the user’s device.
-   Next, replace the last few layers of the base model with untrained layers and ship it to the device.

It is expected that the first few layers would have learned to identify the key facial features.

From there on, training on the user’s face won’t require much data.

But yet again, **you realize that you shall run into the same problems you observed with the binary classification model since the new layers will still be designed around predicting `1` or `0`.**


#### Solution: Contrastive learning using Siamese Networks
At its core, a Siamese network determines whether two inputs are similar.
![](https://www.dailydoseofds.com/content/images/2024/12/image-6.png)

It does this by learning to map both inputs to a shared embedding space (**the blue layer above**):

-   If the distance between the embeddings is LOW, they are similar.
-   If the distance between the embeddings is HIGH, they are dissimilar.

They are beneficial for tasks where the goal is to **compare** two data points rather than to classify them into predefined categories/classes.

This is how it will work in our case:
-   If a pair belongs to the same person, the true label will be 0.
-   If a pair belongs to different people, the true label will be 1.

Create a dataset of face pairs:

![](https://www.dailydoseofds.com/content/images/2024/12/image-7.png)


After creating this data, define a network like this:

![](https://www.dailydoseofds.com/content/images/2024/12/image-8.png)

**Contrastive loss** (defined below) helps us train such a model:

![](https://www.dailydoseofds.com/content/images/2025/01/image-17.png)

where:

-   `y` is the true label.
-   `D` is the distance between two embeddings.
-   `margin` is a hyperparameter, typically greater than 1.

Here’s how this particular loss function helps:

When y=0 (same person), the loss will be:

![](https://www.dailydoseofds.com/content/images/2024/12/image-10.png)

-   The above value will be minimum when D is close to `0`, leading to a low distance between the embeddings.

When `y=1` (different people), the loss will be:

![](https://www.dailydoseofds.com/content/images/2025/01/image-18.png)

-   The above value will be minimum when  `D>margin`, leading to more distance between the embeddings.

This way, we can ensure that:

-   when the inputs are similar, they lie closer in the embedding space.
-   when the inputs are dissimilar, they lie far in the embedding space.

----------

#### Siamese Networks in face unlock

First, you will train the model on several image pairs using contrastive loss.

![](https://www.dailydoseofds.com/content/images/2024/12/image-12.png)

This model (likely after [model compression](https://www.dailydoseofds.com/model-compression-a-critical-step-towards-efficient-machine-learning/)) will be shipped to the user’s device.

During the setup phase, the user will provide facial data, which will create a user embedding:

![](https://www.dailydoseofds.com/content/images/2024/12/image-13.png)

This embedding will be stored in the device’s memory.

Next, when the user wants to unlock the mobile, a new embedding can be generated and compared against the available embedding:

-   Action: Unlock the mobile if the distance is small.


Note that **no further training was required here**, like in the earlier case of binary classification.

Also, what if multiple people want to add their face IDs?

No problem.

![](https://www.dailydoseofds.com/content/images/2024/12/image-14.png)

We can create another embedding for the new user.During unlock, we can compare the incoming user against all stored embeddings.

#### Implementing Contrastive Learning-based Siamese Network

Next, let’s look at the implementation of this model.

For simplicity, we shall begin with a simple implementation utilizing the MNIST dataset. In a future issue, we shall explore the face unlock model.

#### Results

Let’s look at some results using images in the test dataset:

We can generate a similarity score as follows:

![](https://www.dailydoseofds.com/content/images/2024/12/image-16.png)

-   **Image pair #1**: Similarity is high since both images depict the same digit:

![](https://www.dailydoseofds.com/content/images/2024/12/image-17.png)

-   **Image pair #2**: Similarity is low since both images depict different digits:

![](https://www.dailydoseofds.com/content/images/2024/12/image-18.png)

-   **Image pair #3**: Similarity is high since both images depict the same digit:

![](https://www.dailydoseofds.com/content/images/2024/12/image-19.png)

-   **Image pair #4**: Similarity is low since both images depict different digits:

![](https://www.dailydoseofds.com/content/images/2024/12/image-20.png)

Great, it works as expected!

----------

This is the whole idea behind contrastive learning, which is also leveraged in CLIP.

However, CLIP takes this a step further by employing two distinct encoders:

-   One for text and
-   Another for images.

![](https://www.dailydoseofds.com/content/images/2024/12/image-15.png)

**These encoders map their respective modalities (text and image) into a shared multimodal embedding space**. This enables a cross-modal comparison—a text can be compared to an image, or vice versa, to evaluate their semantic similarity, which is a key component of a multimodal RAG system.


----------
REF: https://towardsdatascience.com/clip-model-and-the-importance-of-multimodal-embeddings-1c8f6b13bf72/
### **Applications of CLIP**

1.  **Image Classification and Retrieval**:  
    CLIP links images with natural language descriptions, enabling flexible tasks like searching images using text queries or classifying images without labeled data (zero-shot classification).
    
2.  **Content Moderation**:  
    By analyzing images and paired text, CLIP can detect inappropriate or harmful content in online platforms, enhancing automated moderation capabilities.
    

----------

### **What is CLIP and How Does it Work?**

CLIP is designed for multi-modal learning. It creates an embedding space where **images** and **text** with similar meanings are closer together. Here's a summary of the pseudocode you shared:

1.  **Image and Text Encoders**:
    
    -   `image_encoder`: Can be ResNet or Vision Transformer. Encodes images into feature vectors (`I_f`).
    -   `text_encoder`: Can be CBOW, BERT, or Text Transformer. Encodes text into feature vectors (`T_f`).
2.  **Joint Multimodal Embedding**:
    
    -   Use projection matrices (`W_i`, `W_t`) to map features into a shared embedding space (`I_e`, `T_e`), normalized to unit vectors.
3.  **Similarity Computation**:
    
    -   Pairwise cosine similarities between embeddings (`logits`) measure compatibility between images and text.
4.  **Loss Function**:
    
    -   A symmetric cross-entropy loss trains the model by maximizing similarity for correct pairs and minimizing it for mismatched pairs.
![Architecture of CLIP model (taken from the original paper)](https://towardsdatascience.com/wp-content/uploads/2023/12/1LEc2qQNO6Vumrv5lqSpuhA.png)


### **Step-by-Step Explanation of the Custom CLIP Model**

This implementation follows the CLIP (Contrastive Language-Image Pretraining) framework, which aligns images and text in a shared embedding space. Below is a breakdown of how it works.

----------

## **1. Input Data**

The model processes **batches of aligned image-text pairs**, meaning each image has a corresponding caption.

-   **Image Batch:** `I[n, h, w, c]`
    
    -   A batch of `n` images with height `h`, width `w`, and `c` color channels.
        
-   **Text Batch:** `T[n, l]`
    
    -   A batch of `n` text sequences (captions), where `l` is the length of each sequence.
        

Example:  
For `batch_size = 128`, the model processes **128 images** and **128 corresponding captions** in a single forward pass.

----------

## **2. Feature Extraction**

Two separate encoders extract features from images and text:

-   **Image Encoder**:
    
    ```python
    I_f = models.resnet34(pretrained=True) 
    
    ```
    
    -   Uses **ResNet-34** to extract deep features from images.
        
    -   Outputs a feature vector `I_f` of shape `[n, d_i]`, where `d_i` is the image feature dimension.
        
-   **Text Encoder**:
    
    ```python
    T_f = AutoModel.from_pretrained("distilbert-base-multilingual-cased")
    
    ```
    
    -   Uses **DistilBERT** to extract text embeddings from captions.
        
    -   Outputs a feature vector `T_f` of shape `[n, d_t]`, where `d_t` is the text feature dimension.
        

----------

## **3. Learned Projections**

To ensure both **image** and **text** features map to the same space, we apply **learned projection layers**.

Each feature vector is projected using a two-layer neural network:

```python
class Projection(nn.Module):
    def __init__(self, d_in: int, d_out: int, p: float=0.5) -> None:
        super().__init__()
        self.linear1 = nn.Linear(d_in, d_out, bias=False)  # First projection
        self.linear2 = nn.Linear(d_out, d_out, bias=False) # Second projection
        self.layer_norm = nn.LayerNorm(d_out)  # Normalization
        self.drop = nn.Dropout(p)  # Dropout

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embed1 = self.linear1(x)
        embed2 = self.drop(self.linear2(F.gelu(embed1)))
        embeds = self.layer_norm(embed1 + embed2)  # Residual connection
        return embeds

```

-   **First Linear Layer (`linear1`)**: Maps features from `d_in` to `d_out`.
    
-   **Second Linear Layer (`linear2`)**: Adds another transformation to improve learning.
    
-   **Layer Normalization (`layer_norm`)**: Ensures stability during training.
    
-   **Dropout (`drop`)**: Prevents overfitting.
    

### **Projection Matrices**

-   `W_i[d_i, d_e]` → Maps image features to `d_e`-dimensional space.
    
-   `W_t[d_t, d_e]` → Maps text features to `d_e`-dimensional space.
    

After projection, both **image and text features** are aligned in the same **embedding space**.

----------

## **4. Embedding and Normalization**

Each feature vector is **normalized** to unit length to ensure cosine similarity comparisons are meaningful:

```python
I_e = l2_normalize(np.dot(I_f, W_i), axis=1)
T_e = l2_normalize(np.dot(T_f, W_t), axis=1)

```

-   `I_e`: Normalized image embeddings.
    
-   `T_e`: Normalized text embeddings.
    

This ensures that embeddings lie on a **unit hypersphere**, improving contrastive learning.

----------

## **5. Vision and Text Encoders**

### **Vision Encoder**

Encodes images using **ResNet-34** and projects them into a shared embedding space.

```python
class VisionEncoder(nn.Module):
    def __init__(self, d_out: int) -> None:
        super().__init__()
        base = models.resnet34(pretrained=True)
        d_in = base.fc.in_features  # Feature dimension
        base.fc = nn.Identity()  # Remove classification head
        self.base = base
        self.projection = Projection(d_in, d_out)  # Projection layer

        # Freeze ResNet weights
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        projected_vec = self.projection(self.base(x))
        projection_len = torch.norm(projected_vec, dim=-1, keepdim=True)
        return projected_vec / projection_len  # Normalize

```

-   **Removes the last fully connected layer** of ResNet.
    
-   **Freezes** ResNet weights (only projection layers are trainable).
    
-   **Normalizes** final embeddings.
    

----------

### **Text Encoder**

Encodes captions using **DistilBERT** and projects them into the same embedding space.

```python
class TextEncoder(nn.Module):
    def __init__(self, d_out: int) -> None:
        super().__init__()
        self.base = AutoModel.from_pretrained(Config.text_model)
        self.projection = Projection(Config.transformer_embed_dim, d_out)

        # Freeze BERT weights
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        out = self.base(x)[0]
        out = out[:, 0, :]  # Extract CLS token
        projected_vec = self.projection(out)
        projection_len = torch.norm(projected_vec, dim=-1, keepdim=True)
        return projected_vec / projection_len  # Normalize

```

-   Uses **CLS token** as the sentence representation.
    
-   Freezes **DistilBERT** parameters.
    
-   Only **projection layers are trainable**.
    

----------

## **6. Computing Cosine Similarity**

Once we obtain **image** and **text** embeddings, we compute cosine similarity:

```python
logits = I_e @ T_e.T

```

-   Computes pairwise similarity between all image-text pairs.
    
-   The similarity matrix is used for contrastive learning.
    

----------

## **7. Contrastive Loss Function**

CLIP uses **contrastive loss** to align images and captions:

```python
def CLIP_loss(logits: torch.Tensor) -> torch.Tensor:
    n = logits.shape[1]      # number of samples
    labels = torch.arange(n) # Create labels tensor
    loss_i = F.cross_entropy(logits.transpose(0, 1), labels, reduction="mean")
    loss_t = F.cross_entropy(logits, labels, reduction="mean")
    loss = (loss_i + loss_t) / 2  # Symmetric loss
    return loss

```

-   **Cross-entropy loss** forces correct image-text pairs to be closer while pushing incorrect ones apart.
    
-   The final loss is **symmetric**, meaning it considers **both image and text losses**.
    

----------

## **8. Final Custom CLIP Model**

Combining everything into a **single model**:

```python
class CustomModel(nn.Module):
    def __init__(self, lr: float = 1e-3) -> None:
        super().__init__()
        self.vision_encoder = VisionEncoder(Config.embed_dim)
        self.caption_encoder = TextEncoder(Config.embed_dim)
        self.tokenizer = Tokenizer(AutoTokenizer.from_pretrained(Config.text_model))
        self.lr = lr
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def forward(self, images, text):
        text = self.tokenizer(text).to(self.device)

        image_embed = self.vision_encoder(images)
        caption_embed = self.caption_encoder(text["input_ids"])
        similarity = caption_embed @ image_embed.T

        loss = CLIP_loss(similarity)
        img_acc, cap_acc = metrics(similarity)
        return loss, img_acc, cap_acc

```

-   Uses **VisionEncoder** and **TextEncoder**.
    
-   Tokenizes input text.
    
-   Computes **image-text similarity**.
    
-   Calculates **contrastive loss**.
    
-   Returns **loss and accuracy**.
    

----------

## **Final Summary**

1.  **Extract features** from images (ResNet) and text (DistilBERT).
    
2.  **Project** features into a **shared embedding space**.
    
3.  **Normalize embeddings** to lie on a unit hypersphere.
    
4.  **Compute cosine similarity** between image and text embeddings.
    
5.  **Apply contrastive loss** to align similar pairs while pushing apart dissimilar ones.
    
6.  **Train only the projection layers**, keeping the encoders frozen.
    



- https://gautam75.medium.com/multi-modal-rag-a-practical-guide-99b0178c4fbb